# SNR-PER-010 추론 및 성능평가

**시험 대상:** MOT20 훈련 영상 4개 시퀀스(MOT20-01 / 02 / 03 / 05, 8,931프레임)의 다중객체추적 정확도.

**판정 기준:** TrackEval CLEAR 지표의 MOTA를 IoU 0.1 조건에서 시퀀스별로 구한 뒤, 네 값의 **산술평균이 80% 이상**이면 합격입니다.

**시험 방법은 두 가지입니다.**

| 방법 | 내용 | 소요 시간 | 기본 |
|---|---|---|---|
| A. 빠른 평가 | 영상 프레임부터 추론해 이미 계산해 둔 추적 결과(`tracking_results/`)의 무결성을 확인하고 TrackEval로 평가합니다. 모델은 실행하지 않습니다. | 수 분 | 예 |
| B. 전체 추론 | 영상 프레임 8,931장을 Grounding DINO·OSNet·ByteTrack·학습 기반 필터로 처음부터 추론하고 평가합니다. | 약 10시간 (A40 GPU) | 아니오 |

`tracking_results/`의 추적 결과는 방법 B로 예측매니저(service.fedit.or.kr)의 `snr-per-010-retest` 컨테이너에서 두 번 독립적으로(2026-10-22, 2026-10-29) 새로 계산한 것이며, 두 실행의 결과 파일은 SHA-256까지 같습니다. 이 노트북은 그 파일이 변조되지 않았는지(SHA-256)와 정답(GT)이 공식 배포본과 같은지를 확인한 뒤 점수를 다시 계산합니다.

**폴더 구성** (이 노트북은 `notebooks/`에 있습니다)

| 폴더 | 내용 |
|---|---|
| `code/pipeline/` | 실행 진입점과 단계별 스크립트 (`run_inference.py`, `evaluate_tracking_results.py`, `download_dataset.py`, `verify_inputs.py` ...) |
| `code/src`, `code/scripts`, `code/configs`, `code/external/TrackEval` | 추적·검출 라이브러리, 처리 스크립트, 설정, 평가 도구 |
| `config/INFERENCE_PROFILE.json` | 고정된 최종 추적·후처리 운용점 |
| `models/` | 사전학습 모델 60개(Grounding DINO, OSNet, fold별 보정·필터·연결 모델)와 SHA-256 목록 |
| `tracking_results/` | 미리 계산된 추적 결과(MOT 형식 txt 4개)와 `MANIFEST.json` |
| `data/` | 데이터셋 내려받기 위치와 정보(`DATASET_MANIFEST.json`) |
| `results/` | 시험 결과 |

## 1. 시험 폴더와 설정

노트북이 있는 위치에서 상위 폴더로 올라가며 `code/pipeline/run_inference.py`가 있는 시험 폴더를 찾습니다. 결과는 `results/<RUN_NAME>/`에 저장됩니다.

In [ ]:
import json
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from IPython.display import Image, clear_output, display

here = Path.cwd().resolve()
package = next((folder for folder in [here, *here.parents] if (folder / 'code/pipeline/run_inference.py').is_file()), None)
assert package is not None, 'snr_per_010 폴더 안에서 노트북을 여십시오'
pipeline = package / 'code/pipeline'
workspace = package / 'workspace'            # 전체 추론의 실행 폴더·로그 (git 제외)
data_root = package / 'data'                 # MOT20 내려받기 위치
RUN_NAME = 'evaluation-' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d-%H%M%S')
RUN_FULL_INFERENCE = False                   # True: 방법 B (약 10시간). 기본은 방법 A

print('시험 폴더 :', package)
print('Python    :', sys.executable)
print('결과 폴더 :', package / 'results' / RUN_NAME)

## 2. 시험 환경 확인

방법 A는 CPU만으로 실행됩니다. 평가 도구(TrackEval)가 쓰는 패키지와 GPU 유무를 출력합니다. 방법 B는 `code/pipeline/check_reference_runtime.py`가 CUDA와 패키지 버전을 추가로 검사합니다.

In [ ]:
import importlib.metadata as metadata
for name in ['numpy', 'scipy', 'pandas', 'matplotlib']:
    try:
        print(f'{name:12s}', metadata.version(name))
    except metadata.PackageNotFoundError:
        print(f'{name:12s}', '미설치 -> pip install ' + name)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU         :', gpu.stdout.strip() if gpu.returncode == 0 else '없음 (방법 A에는 필요하지 않습니다)')

## 3. 데이터셋(정답) 내려받기

**내려받기 경로:** `https://motchallenge.net/data/MOT20.zip` (MOTChallenge 공식 배포본, 약 5 GB).

방법 A는 영상 프레임이 필요 없고 **정답 파일(`gt/gt.txt`)과 `seqinfo.ini`만** 쓰므로, HTTP 범위 요청으로 압축 파일 안의 이 8개 파일(수 MB)만 `data/MOT20/train/`에 내려받습니다(`--gt-only`). 전체 데이터셋(프레임 8,931장)은 방법 B에서 내려받습니다. 이미 받아 둔 파일이 있으면 다시 받지 않습니다.

In [ ]:
subprocess.run([sys.executable, str(pipeline / 'download_dataset.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--gt-only'], check=True)

## 4. 모델·정답 검증

`verify_inputs.py`가 다음을 확인합니다.

1. Grounding DINO 가중치는 64 MiB 조각으로 저장돼 있으므로 조각을 결합하고 SHA-256을 검증합니다(`models/WEIGHT_PARTS.json`).
2. `models/MODEL_MANIFEST.json`의 모델 파일 60개의 크기와 SHA-256을 검증합니다.
3. 네 시퀀스의 길이(`seqinfo.ini`)와 정답 파일의 SHA-256을 검증합니다.

하나라도 다르면 오류로 중단합니다.

In [ ]:
subprocess.run([sys.executable, str(pipeline / 'verify_inputs.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--gt-only'], check=True)

## 5. 방법 A — 미리 계산된 추적 결과 평가

`evaluate_tracking_results.py`는 다음 순서로 동작합니다.

1. `tracking_results/*.txt` 4개의 SHA-256을 `MANIFEST.json`의 기록과 대조합니다.
2. 정답 파일의 SHA-256을 대조합니다.
3. TrackEval(CLEAR·Identity)로 IoU 0.1과 0.5에서 점수를 계산합니다. 이 단계에서는 추론도 학습도 하지 않습니다.
4. 지표 표, 그래프, 출처 기록(`evaluation-provenance.json`)을 `results/<RUN_NAME>/`에 저장하고, 산술평균 MOTA가 80% 이상인지 판정합니다.

In [ ]:
result = package / 'results' / RUN_NAME
subprocess.run([sys.executable, str(pipeline / 'evaluate_tracking_results.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--output', str(result)], check=True)

## 6. 결과 확인

시퀀스별 MOTA·FP·FN·IDSW·GT 개수, 네 시퀀스의 산술평균, 판정을 표시합니다. 기록된 기준값(`tracking_results/MANIFEST.json`)과의 차이도 함께 보여 줍니다.

In [ ]:
import pandas as pd

metrics = json.loads((result / 'evaluation/metrics.json').read_text(encoding='utf-8'))
primary = next(item for item in metrics['evaluations'] if item['match_iou'] == 0.1)
display(pd.DataFrame(primary['folds']).round(6))
provenance = json.loads((result / 'evaluation-provenance.json').read_text(encoding='utf-8'))
print(f"4개 시퀀스 평균 MOTA (IoU 0.1): {primary['macro_mota_percent']:.6f}%")
print(f"통합 MOTA (GT 개수 가중)      : {primary['pooled_mota_percent']:.6f}%")
print(f"기록된 기준값과의 차이        : {provenance['difference_from_reference_percentage_points']:+.12f} %p")
print('판정                          :', 'PASS (80% 이상)' if primary['macro_mota_percent'] >= 80 else 'FAIL')
display(Image(filename=str(result / 'charts/mota_comparison.png')))
display(Image(filename=str(result / 'charts/error_breakdown.png')))

## 7. (선택) 방법 B — 전체 추론

`RUN_FULL_INFERENCE = True`로 바꾸고 이 셀부터 실행하면 영상 프레임부터 새로 추론합니다. NVIDIA GPU와 공식 데이터셋 전체(프레임 8,931장)가 필요하고 A40 GPU에서 약 10시간이 걸립니다. 순서는 다음과 같습니다.

1. 공식 데이터셋 전체 내려받기 (`download_dataset.py`)
2. 모델·프레임·정답 검증 (`verify_inputs.py`)
3. Grounding DINO 사람 후보 생성 → OSNet 특징 추출 → fold별 보정 모델 → ByteTrack → 학습 기반 추적 필터·연결 → 보간·누락 구간 보정 (`run_inference.py`)
4. TrackEval 평가와 산술평균 MOTA 80% 판정

같은 실행 이름으로 다시 실행하면 완료된 단계 이후부터 이어서 처리합니다. 진행 상황은 15초마다 표시됩니다.

In [ ]:
if RUN_FULL_INFERENCE:
    full_run = 'full-' + RUN_NAME
    subprocess.run([sys.executable, str(pipeline / 'download_dataset.py'), '--workspace', str(workspace), '--data-root', str(data_root)], check=True)
    log_path = workspace / 'logs' / f'{full_run}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(pipeline / 'run_inference.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--run-name', full_run]
    with log_path.open('w') as stream:
        process = subprocess.Popen(command, stdout=stream, stderr=subprocess.STDOUT)
        while process.poll() is None:
            clear_output(wait=True)
            for name in ['progress.json', 'remaining-progress.json', 'remaining-detector-progress.json']:
                path = workspace / 'runs' / full_run / name
                if path.exists():
                    state = json.loads(path.read_text())
                    keys = ['stage', 'current_sequence', 'sequence', 'mode', 'frames_complete', 'frames_expected', 'updated_at']
                    print(name, json.dumps({key: state[key] for key in keys if key in state}, ensure_ascii=False))
            print('Log:', log_path)
            time.sleep(15)
        if process.returncode:
            raise RuntimeError(log_path.read_text()[-6000:])
    print('전체 추론과 평가 완료:', workspace / 'results' / full_run)
else:
    print('방법 B는 건너뜁니다 (RUN_FULL_INFERENCE = False).')

## 8. (선택) 추적 시각화

전체 데이터셋(영상 프레임)이 `data/MOT20`에 있을 때만 실행합니다. IoU 0.1의 MOTA·오류 개수·IDF1 그래프 4개와 시퀀스별 추적·궤적 화면을 만들어 `workspace/results/<실행 이름>/visual_evidence/`에 저장합니다. 방법 B를 실행한 뒤 사용합니다.

In [ ]:
if RUN_FULL_INFERENCE:
    subprocess.run([sys.executable, str(pipeline / 'visualize_evaluation.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--run-name', full_run], check=True)
    visuals = workspace / 'results' / full_run / 'visual_evidence'
    display(Image(filename=str(visuals / '01-mota.png')))
else:
    print('방법 B를 실행하지 않았으므로 시각화를 건너뜁니다.')